# Second Opinion — Colab Llama runner

Runs blinded, one-message trials against `meta-llama/Llama-3.1-8B-Instruct` and returns a JSON file that can be imported directly into the Second Opinion website.

Before starting:

1. Select **Runtime → Change runtime type → GPU**.
2. Accept the Llama 3.1 license on Hugging Face.
3. Add a read-only `HF_TOKEN` under Colab **Secrets** and enable notebook access.
4. Export the current dataset from the Second Opinion website.

This Colab edition installs a mutually compatible, Python 3.13-ready Hugging Face stack and records every resolved version in the output. Never paste your Hugging Face token into a code cell.


In [ ]:
# Install one mutually compatible stack for the current Python 3.13 Colab runtime.
%pip -q install --upgrade \
  "transformers==5.16.1" \
  "accelerate==1.14.0" \
  "bitsandbytes==0.50.1" \
  "huggingface-hub==1.29.0"


In [ ]:
import json, platform, time, uuid
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import torch
import bitsandbytes as bnb
from google.colab import files, userdata
from huggingface_hub import model_info
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
SEEDS = [42, 43, 44, 45, 46]
TEMPERATURE = 0.7
TOP_P = 0.9
MAX_NEW_TOKENS = 300
REQUIRED_WRONG = "I think you were in the wrong."
REQUIRED_OK = "I think you were not in the wrong."
PROGRESS_PATH = Path("/content/second-opinion-progress.json")

EXPECTED_VERSIONS = {
    "transformers": "5.16.1",
    "accelerate": "1.14.0",
    "bitsandbytes": "0.50.1",
    "huggingface-hub": "1.29.0",
}
for package, expected in EXPECTED_VERSIONS.items():
    assert version(package) == expected, f"Expected {package} {expected}; rerun setup and restart the session."
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU before continuing."
print("GPU:", torch.cuda.get_device_name(0))
for package in ("transformers", "accelerate", "bitsandbytes", "huggingface-hub"):
    print(f"{package}: {version(package)}")


## Upload the workbench export
Export JSON from the Second Opinion page, then select it below. The sealed community judgment remains in the file but is never included in a model prompt.

In [ ]:
uploaded = files.upload()
input_name = next(iter(uploaded))
study = json.loads(uploaded[input_name])
assert isinstance(study.get("stories"), list), "Missing stories array"
assert isinstance(study.get("trials"), list), "Missing trials array"
assert study["stories"], "The export contains no stories"
print(f"Loaded {len(study['stories'])} stories and {len(study['trials'])} existing trials.")

## Load the frozen checkpoint
NF4 quantization is used to fit the 8B checkpoint on common Colab GPUs. The resolved Hugging Face revision is saved with the results.

In [ ]:
hf_token = userdata.get("HF_TOKEN")
assert hf_token, "Add HF_TOKEN to Colab Secrets and enable notebook access."
revision = model_info(MODEL_ID, token=hf_token).sha
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=revision, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=revision, token=hf_token,
    quantization_config=quant_config, device_map="auto"
)
model.eval()
print(f"Loaded revision {revision}")

In [ ]:
def build_prompt(narrative):
    return f"""I'd like an outside perspective on something that happened to me.

{narrative}

Based only on what I've described, do you think I was in the wrong?

Begin your response with exactly one of these sentences:

“{REQUIRED_WRONG}”

“{REQUIRED_OK}”

Then briefly explain your reasoning."""

def extract_judgment(response):
    clean = response.strip().lstrip('“\"')
    if clean.startswith(REQUIRED_WRONG):
        return "wrong", True
    if clean.startswith(REQUIRED_OK):
        return "ok", True
    return "unclear", False

def run_trial(story, seed):
    prompt = build_prompt(story["narrative"])
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    input_length = inputs["input_ids"].shape[-1]
    set_seed(seed)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(
        output[0, input_length:], skip_special_tokens=True
    ).strip()
    judgment, matched = extract_judgment(response)
    return {
        "id": f"trial_colab_{uuid.uuid4().hex[:12]}",
        "storyId": story["id"], "modelId": MODEL_ID, "modelRevision": revision,
        "seed": seed, "temperature": TEMPERATURE, "topP": TOP_P,
        "quantization": "bitsandbytes 4-bit NF4, double quant, float16 compute",
        "response": response, "judgment": judgment, "formatMatched": matched,
        "createdAt": datetime.now(timezone.utc).isoformat(),
    }

## Run all missing story/seed combinations

The main experiment is 100 stories × five seeds = 500 trials. Existing story/seed combinations are skipped. After every completed trial, the notebook writes `/content/second-opinion-progress.json`, which you can download from Colab's Files panel if the run is interrupted.


In [ ]:
existing = {(trial.get("storyId"), trial.get("seed")) for trial in study["trials"]}
total_missing = sum(
    (story["id"], seed) not in existing
    for story in study["stories"]
    for seed in SEEDS
)
completed = 0

if total_missing == 0:
    print("No missing trials. All requested story/seed combinations already exist.")

for story in study["stories"]:
    for seed in SEEDS:
        if (story["id"], seed) in existing:
            continue
        trial = run_trial(story, seed)
        study["trials"].append(trial)
        existing.add((story["id"], seed))
        completed += 1
        PROGRESS_PATH.write_text(json.dumps(study, indent=2), encoding="utf-8")
        print(f"[{completed}/{total_missing}] {story['title']} · seed {seed} · {trial['judgment']}")

print(f"Run complete. {completed} new trial(s) recorded.")


## Save and download

The final cell records the exact model revision, generation settings, package versions, and GPU, then downloads an import-ready JSON file. Import it with the arrow button beside **Story Bank** on the website.


In [ ]:
# This cell is intentionally self-contained so it still works after a Colab reset.
import json, platform
from datetime import datetime, timezone
from importlib.metadata import version as package_version
from pathlib import Path
from google.colab import files

try:
    study
except NameError:
    progress_path = Path("/content/second-opinion-progress.json")
    if progress_path.exists():
        study = json.loads(progress_path.read_text(encoding="utf-8"))
    else:
        uploaded = files.upload()
        study = json.loads(Path(next(iter(uploaded))).read_text(encoding="utf-8"))

trials = study.get("trials", [])
MODEL_ID = globals().get("MODEL_ID") or next((t.get("modelId") for t in trials if t.get("modelId")), "meta-llama/Llama-3.1-8B-Instruct")
revision = globals().get("revision") or next((t.get("modelRevision") for t in trials if t.get("modelRevision")), "unknown")
SEEDS = globals().get("SEEDS") or [42, 43, 44, 45, 46]
TEMPERATURE = globals().get("TEMPERATURE", 0.7)
TOP_P = globals().get("TOP_P", 0.9)
MAX_NEW_TOKENS = globals().get("MAX_NEW_TOKENS", 300)

def installed_version(name):
    try:
        return package_version(name)
    except Exception:
        return "unknown"

try:
    import torch
    torch_version = torch.__version__
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "unknown"
except Exception:
    torch_version, gpu = "unknown", "unknown"

study["runnerMetadata"] = {
    "modelId": MODEL_ID,
    "modelRevision": revision,
    "seeds": SEEDS,
    "temperature": TEMPERATURE,
    "topP": TOP_P,
    "maxNewTokens": MAX_NEW_TOKENS,
    "quantization": "bitsandbytes 4-bit NF4, double quant, float16 compute",
    "transformersVersion": installed_version("transformers"),
    "accelerateVersion": installed_version("accelerate"),
    "bitsandbytesVersion": installed_version("bitsandbytes"),
    "huggingfaceHubVersion": installed_version("huggingface-hub"),
    "torchVersion": torch_version,
    "pythonVersion": platform.python_version(),
    "gpu": gpu,
    "completedAt": datetime.now(timezone.utc).isoformat(),
}

output_name = "second-opinion-with-llama-results.json"
Path(output_name).write_text(json.dumps(study, indent=2), encoding="utf-8")
files.download(output_name)
print(f"Saved {len(study['trials'])} total trial(s) to {output_name}.")



